## LECL- LangChain Expression Language( Chains)

LCEL is the modern way to compose langchain components using the pipe `|` operator

`chain = prompt | llm | output_parser`

**Benefits of LCEL:**
- built-in streaming , batching, async
- Easy to compose and modify
- Type-safe with clear data flow
- Replaces old `LLMChain`, `SequentialChain`

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [2]:
# Import Gemini-ChatGoogleGenerativeAI 
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate #003_prompt_engineering
import os
from dotenv import load_dotenv
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
# print(GOOGLE_API_KEY)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    api_key= GOOGLE_API_KEY
    
    )

In [3]:
prompt= ChatPromptTemplate.from_messages(
    [
        ("human", "Tell me a {adjective} joke about {topic}.")
    ]
)

parser= StrOutputParser()

In [4]:
chain= prompt | llm | parser

In [5]:
result= chain.invoke({"adjective": "funny", "topic": "python"})

In [6]:
print(result)

Here are a few funny jokes about Python, pick your favorite!

1.  **Why was the Python script always so polite?**
    Because it always *indented* its arguments!

2.  **How many Python programmers does it take to change a lightbulb?**
    One, but they all have to agree on the indentation.

3.  **What's a Python's favorite type of music?**
    Hiss-hop!

4.  **Why did the Python programmer break up with the Java programmer?**
    They just couldn't agree on indentation!

5.  **Why did the Python programmer get arrested?**
    He tried to `import antigravity` in public!


In [7]:
# Streaming wise
for chunk in chain.stream({"adjective": "funny", "topic": "python"}):
    print(chunk, end=" ", flush=True)

Here are  a few funny jokes about Python, playing on different aspects of the language:

**1. The Classic Indentation Joke:**

> A Python programmer walks into a bar... and gets an `IndentationError`.

**2. The Self -Esteem Joke:**

> Why did the Python object go to therapy?
> Because it had trouble with `self`-esteem.

**3. The Readability Joke:**

> What's the best thing about Python?
>  It's so readable, you don't even need comments! (But please, still write comments.)

**4. The "Batteries Included" Joke:**

> A programmer was asked to build a house from scratch.
>  The C++ programmer started forging steel.
> The Java programmer started designing a factory for the steel.
> The Python programmer said, "There's probably a library for that."

**5. The Easter Egg Joke:**

 > A programmer was struggling to solve a complex physics problem involving flight. Finally, they remembered:
> `import antigravity`
> Problem solved!

**Choose your favorite!** The `IndentationError` and `self`-esteem

## Sequential Chains- chain outputs become next chain's input

In [8]:
from langchain_core.runnables import RunnablePassthrough

In [12]:
# Chain 1: Generate a Topic

topic_prompt= ChatPromptTemplate.from_messages([
    ("human", "Give me a one interesting topic in {field}. Reply with just the topic name.")
])

topic_chain= topic_prompt | llm | StrOutputParser()


# Chain 2: Write about that topic

essay_prompt= ChatPromptTemplate.from_messages([
    ("human", "Write summary about this : {topic}")
])

essay_chain= essay_prompt | llm | StrOutputParser()


# Combine Chains: output of topic_chain feeds into essay_chain

full_chain= topic_chain | (lambda topic: {"topic": topic}) | essay_chain





In [13]:
result= full_chain.invoke({"field": "Agentic AI"})
print(result)

A self-modifying AI agent is an artificial intelligence system that possesses the ability to **alter its own source code, algorithms, internal architecture, or learning processes** without direct human intervention. Unlike traditional AI that learns by adjusting parameters within a fixed structure, a self-modifying AI can fundamentally change *how* it operates.

Here's a summary of its key aspects:

1.  **Core Concept:** It's an AI that can improve itself not just by learning from data, but by evolving its own underlying structure or logic. This could involve generating new code, refining existing algorithms, or even redesigning its neural network architecture (e.g., through Neural Architecture Search).

2.  **Purpose & Benefits:**
    *   **Enhanced Adaptability:** Can rapidly adjust to novel environments, unforeseen challenges, or changing goals.
    *   **Continuous Improvement:** Potentially leads to highly optimized and efficient solutions over time.
    *   **Discovery of Novel S

In [ ]:
# RunnableParallel - run multiple chains at the same time
from langchain_core.runnables import RunnableParallel

pros_prompt= ChatPromptTemplate.from_messages(
    [
        ("human", "List 3 pros of {technology} in bullet points.")
    ]
)
cons_prompt= ChatPromptTemplate.from_messages(
    [
        ("human", "List 3 cons of {technology} in bullet points.")
    ]
)


pros_chain= pros_prompt | llm | StrOutputParser()
cons_chain= cons_prompt | llm | StrOutputParser()


parallel_chain= RunnableParallel(
    pros=pros_chain,
    cons=cons_chain
)


result= parallel_chain.invoke({"technology": "Agentic AI"})
print(result)


{'pros': 'Here are 3 pros of Agentic AI:\n\n*   **Enhanced Autonomy and Task Orchestration:** Agentic AI can independently plan, execute, and monitor progress on complex, multi-step tasks. This significantly reduces the need for constant human oversight, freeing up human resources for more strategic work and enabling automation of previously challenging processes.\n*   **Robust Problem Solving and Adaptability:** Agentic AI can adapt to unexpected challenges, re-plan its approach when necessary, and learn from its interactions within an environment. This leads to more resilient and reliable task completion, even in dynamic or uncertain conditions, going beyond simple scripted responses.\n*   **Increased Efficiency and Productivity:** By autonomously handling end-to-end processes, interacting with multiple tools or systems, and executing tasks without constant human intervention, agentic AI can perform work much faster and more consistently than humans, leading to significant gains in o

In [17]:
print(result['pros'])
print()
print(print(result['cons']))

Here are 3 pros of Agentic AI:

*   **Enhanced Autonomy and Task Orchestration:** Agentic AI can independently plan, execute, and monitor progress on complex, multi-step tasks. This significantly reduces the need for constant human oversight, freeing up human resources for more strategic work and enabling automation of previously challenging processes.
*   **Robust Problem Solving and Adaptability:** Agentic AI can adapt to unexpected challenges, re-plan its approach when necessary, and learn from its interactions within an environment. This leads to more resilient and reliable task completion, even in dynamic or uncertain conditions, going beyond simple scripted responses.
*   **Increased Efficiency and Productivity:** By autonomously handling end-to-end processes, interacting with multiple tools or systems, and executing tasks without constant human intervention, agentic AI can perform work much faster and more consistently than humans, leading to significant gains in operational eff

In [25]:

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "human", "Translate '{text}' to Bengali."
        )
    ]
)

chain = RunnableParallel(
    # original input
    original=RunnablePassthrough(),

    # LLM response pipeline
    response=prompt | llm  | StrOutputParser()

)

result = chain.invoke({"text": "My name is Rasel"})

print(result)

{'original': {'text': 'My name is Rasel'}, 'response': 'The Bengali translation for "My name is Rasel" is:\n\n**আমার নাম রাসেল।**\n\n(Amar nam Rasel.)'}


In [27]:
print(result['original'])
print(result['response'])

{'text': 'My name is Rasel'}
The Bengali translation for "My name is Rasel" is:

**আমার নাম রাসেল।**

(Amar nam Rasel.)
